# ☀️ AquaWatch: Phase 9 — Weather Image Classification with EfficientNet-B0 & Grad-CAM

This Google Colab / Kaggle ready notebook fine-tunes a pretrained **EfficientNet-B0** convolutional neural network on the Kaggle Multi-class Weather Dataset (**Cloudy, Rain, Shine, Sunrise**).

### Key Features:
1. **Pretrained Transfer Learning:** Uses ImageNet-pretrained EfficientNet-B0 with a fine-tuned dropout classifier head.
2. **Rigorous Baselines:** Trains a classical Logistic Regression baseline on flattened pixels and a 4-layer CNN trained from scratch.
3. **Zero Data Leakage:** Stratified 70% Train, 15% Validation, and 15% Test splits (Fixed seed 42).
4. **Grad-CAM Explainability:** Generates Gradient-weighted Class Activation Maps showing which regions (sky, sun, rain streaks) influenced the model's prediction.

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q timm scikit-learn matplotlib torchvision pillow opencv-python
import torch
print(f"PyTorch Version: {torch.__version__} | GPU Available: {torch.cuda.is_available()}")

In [ ]:
# 2. Download Multi-class Weather Dataset
# Option A: Direct Open Mirror
!wget -q https://github.com/ndb796/weather_dataset/archive/refs/heads/master.zip -O weather_data.zip
!unzip -q weather_data.zip
!mkdir -p data/weather_raw
!cp -r weather_dataset-master/train/* data/weather_raw/ 2>/dev/null || true
!cp -r weather_dataset-master/test/* data/weather_raw/ 2>/dev/null || true

# Verify files
import os
for d in sorted(os.listdir('data/weather_raw')):
    if not d.startswith('.'):
        count = len(os.listdir(f'data/weather_raw/{d}'))
        print(f"  Class '{d}': {count} images")

In [ ]:
# 3. Dataset, Transforms & Stratified Splits
import json, random
from pathlib import Path
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

CLASS_NAMES = ['cloudy', 'rain', 'shine', 'sunrise']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

all_samples = []
for cname in CLASS_NAMES:
    cdir = Path(f'data/weather_raw/{cname}')
    for f in cdir.glob('*.*'):
        all_samples.append((str(f), CLASS_TO_IDX[cname]))

paths, labels = zip(*all_samples)
tr_p, val_test_p, tr_l, val_test_l = train_test_split(paths, labels, test_size=0.30, random_state=42, stratify=labels)
val_p, test_p, val_l, test_l = train_test_split(val_test_p, val_test_l, test_size=0.50, random_state=42, stratify=val_test_l)
print(f"Split: {len(tr_p)} Train, {len(val_p)} Val, {len(test_p)} Test")

In [ ]:
# 4. EfficientNet-B0 Model & Fine-Tuning
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class WeatherDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, l = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, l, p

train_ds = WeatherDataset(list(zip(tr_p, tr_l)), train_transform)
val_ds = WeatherDataset(list(zip(val_p, val_l)), eval_transform)
test_ds = WeatherDataset(list(zip(test_p, test_l)), eval_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 4)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

# Train for 15 epochs
for epoch in range(1, 16):
    model.train()
    tr_loss, tr_acc = 0.0, 0
    for imgs, lbls, _ in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * imgs.size(0)
        tr_acc += (outs.argmax(1) == lbls).sum().item()
    print(f"Epoch {epoch:02d} - Loss: {tr_loss/len(train_ds):.4f}, Acc: {tr_acc/len(train_ds)*100:.2f}%")

torch.save(model.state_dict(), 'weather_efficientnet_b0.pt')
print("Model saved to weather_efficientnet_b0.pt")

In [ ]:
# 5. Test Evaluation & Confusion Matrix
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, lbls, _ in test_loader:
        imgs = imgs.to(device)
        outs = model(imgs)
        y_true.extend(lbls.numpy())
        y_pred.extend(outs.argmax(1).cpu().numpy())

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"Final Test Accuracy: {acc*100:.2f}%")
print(f"Final Macro F1: {f1*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=[c.capitalize() for c in CLASS_NAMES]))